<a href="https://colab.research.google.com/github/sunonmountain/Revenue-Integrity-Engine/blob/main/05_Autonomous_High_Ticket_Outreach_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install -U google-genai pandas pydantic

import pandas as pd
import json
import time
import os
from google import genai
from google.genai import types
from google.colab import userdata
from pydantic import BaseModel, Field

# --- 1. CONFIGURATION & SECRETS ---
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print("✅ Gemini API Key retrieved from Secrets.")
except Exception:
    GEMINI_API_KEY = "PASTE_YOUR_KEY_HERE"

MODEL_ID = "gemini-3-flash-preview"

# --- 2. DATA MODELS ---
class OutreachDraft(BaseModel):
    subject_line: str = Field(description="Short, curiosity-inducing subject line (3-5 words).")
    email_body: str = Field(description="Direct, personalized email body under 125 words.")
    ai_strategy: str = Field(description="Brief explanation of the psychological angle used.")

# --- 3. NUMERIC ROI TRACKER ---
class OutreachROI:
    def __init__(self):
        self.count_emails_drafted = 0
        self.start_time = time.time()
        self.human_time_saved_hours = 0.0
        self.hourly_rate_gbp = 25.0 # Standard BDR/SDR rate

    def log_action(self):
        self.count_emails_drafted += 1
        # 15 minutes per high-ticket lead = 0.25 hours
        self.human_time_saved_hours += 0.25

    def get_report(self):
        execution_time_seconds = round(time.time() - self.start_time, 2)
        est_labour_savings_gbp = round(self.human_time_saved_hours * self.hourly_rate_gbp, 2)

        return {
            "Total Emails Drafted": self.count_emails_drafted,
            "Execution Time Seconds": execution_time_seconds,
            "Human Time Saved Hours": round(self.human_time_saved_hours, 2),
            "Est Labour Savings GBP": est_labour_savings_gbp,
            "Infrastructure Cost GBP": 0.00, # Free tier logic
            "Net ROI GBP": est_labour_savings_gbp
        }

# --- 4. THE OUTREACH ENGINE ---
class HighTicketOutreachEngine:
    def __init__(self, api_key):
        self.client = genai.Client(api_key=api_key)
        self.roi = OutreachROI()
        # YOUR BRAND ASSETS
        self.value_prop = "I build autonomous lead integrity systems that eliminate revenue leaks."
        self.proof_point = "Automated research for a SaaS team, reclaiming 12 hours of their week."

    def craft_email(self, first_name, company, trigger, hook):
        system_instruction = (
            "You are an Elite Sales Copywriter. Write a 1-to-1 email that is brief, "
            "direct, and uses a peer-to-peer tone. Rules: 1. NO 'I hope this finds you well.' "
            "2. Lead with the Research Hook. 3. Be concise. 4. Use a soft CTA (no calendar links)."
        )

        user_prompt = (
            f"Write an email to {first_name} at {company}.\n"
            f"Research Context (Trigger): {trigger}\n"
            f"Specific Hook to use: {hook}\n"
            f"My Offer: {self.value_prop}\n"
            f"Social Proof: {self.proof_point}"
        )

        response = self.client.models.generate_content(
            model=MODEL_ID,
            contents=user_prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                response_mime_type="application/json",
                response_schema=OutreachDraft,
                thinking_config=types.ThinkingConfig(thinking_level="LOW")
            )
        )
        self.roi.log_action()
        return json.loads(response.text)

# --- 5. EXECUTION PIPELINE ---
def run_week_5(demo_mode=True):
    input_file = 'Week4_Gold_Intent_Data.csv'
    if not os.path.exists(input_file):
        print(f"❌ Error: {input_file} not found. Ensure Week 4 completed successfully.")
        return

    df = pd.read_csv(input_file)
    engine = HighTicketOutreachEngine(GEMINI_API_KEY)

    # FILTER: Target leads that have the 'Outreach_Hook' from Week 4
    eligible = df[df['Outreach_Hook'].notna()]
    targets = eligible.head(5) if demo_mode else eligible

    print(f"🚀 Generating {len(targets)} personalized drafts...")

    for index, row in targets.iterrows():
        print(f"   Drafting: {row['First_Name']} @ {row['Company']}...")
        try:
            draft = engine.craft_email(
                row['First_Name'],
                row['Company'],
                row['Trigger_Event'],
                row['Outreach_Hook']
            )

            # Map back to CSV
            df.at[index, 'Email_Subject'] = draft['subject_line']
            df.at[index, 'Email_Body'] = draft['email_body']
            df.at[index, 'AI_Strategy_Logic'] = draft['ai_strategy']

            time.sleep(1.2) # Rate limit safety

        except Exception as e:
            print(f"   ❌ Failed at index {index}: {str(e)}")

    # Final Export
    output_file = 'Week5_Final_Outreach_Leads.csv'
    df.to_csv(output_file, index=False)

    print("\n" + "="*35)
    print("WEEK 5 FINAL ROI REPORT (NUMERIC)")
    print(json.dumps(engine.roi.get_report(), indent=4))
    print("="*35)

run_week_5(demo_mode=True)

✅ Gemini API Key retrieved from Secrets.
🚀 Generating 5 personalized drafts...
   Drafting: Gavin @ Techflow Solutions...
   Drafting: Gavin @ Techflow Solutions...
   Drafting: Sarah @ Cloudbridge Uk...
   Drafting: Alistair @ Finscale Ltd...
   Drafting: Priya @ Greenmetrics...

WEEK 5 FINAL ROI REPORT (NUMERIC)
{
    "Total Emails Drafted": 5,
    "Execution Time Seconds": 26.41,
    "Human Time Saved Hours": 1.25,
    "Est Labour Savings GBP": 31.25,
    "Infrastructure Cost GBP": 0.0,
    "Net ROI GBP": 31.25
}
